# Entrenar a Ultra Yo con tus conversaciones (LoRA, gratis)

Este cuaderno ajusta el cerebro ligero de Ultra (Gemma 4 E2B) con lo que habéis hablado, y lo convierte en un archivo `.litertlm` que el teléfono puede usar.

**Antes de empezar**
1. En Colab: *Entorno de ejecución → Cambiar tipo → GPU T4*.
2. Crea una cuenta gratis en huggingface.co, entra en https://huggingface.co/google/gemma-4-E2B-it y acepta la licencia. Luego crea un token de lectura en *Settings → Access Tokens*.
3. Copia del teléfono el archivo `Documentos/Ultra Yo/entrenamiento/conversaciones.jsonl`.

**Aviso sincero:** el entrenamiento cabe en la T4 gratuita. La conversión final a `.litertlm` usa mucha memoria RAM y puede quedarse corta en Colab gratis (unos 12 GB). Si se corta, ejecuta el mismo cuaderno en Kaggle (también gratis, con 30 GB de RAM). Este proceso no se ha probado todavía de principio a fin con tus datos.

In [ ]:
!pip install -q "transformers>=5.5.0" peft bitsandbytes accelerate datasets litert-torch huggingface_hub
!free -h

In [ ]:
from huggingface_hub import login
login()  # pega aquí tu token de Hugging Face

## 1. Sube tus conversaciones
Con menos de unas 200 conversaciones el cambio apenas se notará. Las que vienen del modo experto (`origen: experto`) valen el doble: son respuestas mejores que Ultra aprende a imitar.

In [ ]:
from google.colab import files
subido = files.upload()  # elige conversaciones.jsonl
import json
filas = [json.loads(l) for l in open("conversaciones.jsonl", encoding="utf-8") if l.strip()]
# Las respuestas del experto se repiten para que pesen más.
datos = []
for f in filas:
    datos.append({"messages": f["messages"]})
    if f.get("origen") == "experto":
        datos.append({"messages": f["messages"]})
print(len(filas), "conversaciones,", len(datos), "ejemplos")

## 2. Cargar Gemma 4 E2B en 4 bits (QLoRA) y entrenar

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

BASE = "google/gemma-4-E2B-it"
tok = AutoTokenizer.from_pretrained(BASE)
modelo = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)
modelo = prepare_model_for_kbit_training(modelo)

# En Gemma 4 los nombres de las capas pueden llevar ".linear" al final: se buscan los que existen.
objetivos = sorted({n.split(".")[-2] if n.endswith(".linear") else n.split(".")[-1]
                    for n, _ in modelo.named_modules()
                    if any(n.endswith(k) or n.endswith(k + ".linear") for k in ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
                    and "vision" not in n and "audio" not in n})
print("Capas a ajustar:", objetivos)
modelo = get_peft_model(modelo, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=objetivos, task_type="CAUSAL_LM"))
modelo.print_trainable_parameters()

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

def preparar(ej):
    # Mismo formato de chat que usa el modelo (el mismo final de turno que verá en el teléfono).
    texto = tok.apply_chat_template(ej["messages"], tokenize=False)
    return tok(texto, truncation=True, max_length=1024)

ds = Dataset.from_list(datos).map(preparar, remove_columns=["messages"]).shuffle(seed=1)
args = TrainingArguments(
    output_dir="salida", per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=2, learning_rate=2e-4, warmup_ratio=0.05, logging_steps=10,
    fp16=True, gradient_checkpointing=True, save_strategy="no", report_to=[],
)
Trainer(model=modelo, args=args, train_dataset=ds,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False)).train()
modelo.save_pretrained("adaptador")  # se guarda antes de mezclar: luego no se puede separar

## 3. Probar antes de convertir
Si aquí contesta mal, no sigas: hacen falta más o mejores conversaciones.

In [ ]:
pregunta = [{"role": "user", "content": "¿Quién eres y en qué me ayudas?"}]
entrada = tok.apply_chat_template(pregunta, add_generation_prompt=True, return_tensors="pt").to(modelo.device)
print(tok.decode(modelo.generate(entrada, max_new_tokens=120)[0][entrada.shape[1]:], skip_special_tokens=True))

## 4. Mezclar el ajuste con el modelo completo

In [ ]:
import gc, os, shutil
del modelo; gc.collect(); torch.cuda.empty_cache()
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True)
mezclado = PeftModel.from_pretrained(base, "adaptador").merge_and_unload()
mezclado.save_pretrained("mezclado", safe_serialization=True)
tok.save_pretrained("mezclado")
# Con transformers 5 a veces falta tokenizer.model, y el conversor lo necesita.
if not os.path.exists("mezclado/tokenizer.model"):
    from huggingface_hub import hf_hub_download
    shutil.copy(hf_hub_download(BASE, "tokenizer.model"), "mezclado/tokenizer.model")
del base, mezclado; gc.collect(); torch.cuda.empty_cache()

## 5. Convertir a `.litertlm`
Se usa la plantilla de chat de la versión oficial para teléfonos (`litert-community`); con la que trae el modelo, LiteRT-LM falla al empezar a hablar ("map has no method named get").

In [ ]:
!litert-torch export_hf --help | head -40
!litert-torch export_hf \
    --model=./mezclado \
    --output_dir=./ultra_litertlm \
    --externalize_embedder \
    --jinja_chat_template_override=litert-community/gemma-4-E2B-it-litert-lm
!ls -lh ultra_litertlm

## 6. Pasarlo al teléfono
Descarga el `.litertlm` (o súbelo a tu Google Drive) y cópialo en el teléfono a **Documentos/Ultra Yo/modelos/**. Después, en Ultra: *Ajustes → Cerebro → Propio → Usar este*.

Si no arranca, Ultra vuelve sola al cerebro ligero y te dice el error. El modelo convertido puede no traer la parte que ve fotos: es normal, hablará pero no verá imágenes.